In [ ]:
# conda activate genomic_tools

import sys

sys.path.append("code")

from parse_gtf import * 

### Plot distro of corrs

In [ ]:
fdr_thresh = 0.05
ct_cols = [c for c in psi_fdr_df.columns if c != 'Gene']

n_cols = 2
n_rows = int(np.ceil(len(ct_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 2.5 * n_rows), dpi=200, sharex=True)  # <-- shared x-axis

axes_flat = axes.flatten()

for i, ct in enumerate(ct_cols):
    ax = axes_flat[i]
    observed = psi_corr_df[ct].values
    null = perm_corr_results[ct].flatten()
    sig_mask = (psi_fdr_df[ct] < fdr_thresh).values

    ax.tick_params(labelbottom=True)
    ax.hist(null, bins=100, color='lightgrey', density=True, label='Null (permuted)')
    ax.hist(observed, bins=100, color='steelblue', alpha=0.6, density=True, label='Observed (all)')
    ax.hist(observed[sig_mask], bins=100, color='salmon', alpha=0.8, density=True, 
            label=f'Significant (FDR < {fdr_thresh})')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(ct)
    ax.set_ylabel('Density')
    ax.set_xlabel('Spearman r')
    ax.legend(fontsize=7)

# hide any unused axes
for j in range(len(ct_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig('figures/correlation_distributions.pdf', dpi=200)
plt.close()

### Plot cell type-specific SE bar plots

In [ ]:
outdir = "figures"

for ascending in (True, False):
    plot_ctype_SEs(
        ctype_specific_SEs_strict[str(ascending)],
        psi_corr_df,
        psi_fdr_df.drop(columns='Gene'),
        expr_corr_df,
        SE_coords_df,
        psi_rank_centered,
        ct_ranks_dict,
        outdir=outdir,
        psi_ci_lower_df=psi_ci_lower_df, 
        psi_ci_upper_df=psi_ci_upper_df,
        ascending=ascending,
        n_boot=None
    )